# dmg777k: preprocessing

This notebook prepares the dmg777k dataset for the experiments of Milestone 3. It keeps the authors' pipeline and fixes only the bugs found in the EDA.

**Before running:** prepare the data as in `data/README.md`. The same steps run without the notebook with `uv run python -m preprocessing.pipeline dmg777k`.

In [1]:
import pandas as pd

from preprocessing import pipeline
from preprocessing.geometry import geodesic_km

pd.set_option("display.max_colwidth", 70, "display.width", 160)


def short(iri: str) -> str:
    """Last part of an IRI, for display only; the saved tables keep full IRIs."""
    return iri.rstrip("/").rsplit("/", 1)[-1].rsplit("#", 1)[-1]


r = pipeline.run("dmg777k")
print(f"{len(r.dataset.triples):,} triples loaded, {len(r.triples):,} kept")

777,124 triples loaded, 776,920 kept


## 1. Preparing the data

### 1.1 The authors' pipeline, with fixes

We keep the authors' steps: load the triples, convert the 371 RD New outlines to WGS84, build the graph and find the geometries.

We change five things:

- remove missing values (empty text, placeholders and empty geometries),
- identify a node by its text and kind (entity, or the literal's type), not by text alone,
- treat every WKT-typed literal as a geometry,
- write WKT keywords in capitals (`Point` becomes `POINT`),
- keep coordinates in (longitude, latitude) order.

**Missing values** removed, grouped by text:

In [2]:
removed = r.placeholders_removed.assign(predicate=r.placeholders_removed["predicate"].map(short))
print(f"Triples removed: {removed['triples'].sum():,}")
removed.groupby("object")["triples"].sum().sort_values(ascending=False).to_frame()

Triples removed: 204


,triples
object,
,204


The only missing value is the empty text (204 triples). The authors' `dropna()` removes the same triples, so the graph keeps 776,920 triples, as in the paper.

**Node identity**: by text (authors) and by text and kind (ours):

In [3]:
r.identity

,By text (authors),By text and kind (ours)
Nodes,332194,341269
Self-loops,8396,0
Nodes where a literal is merged into an entity,8396,0
Nodes where literals of different types are merged,632,0


Identifying nodes by text alone merges 9,075 nodes. Every labelled monument is merged with its `seeAlso` literal, which has the same text, so it gets a self-loop. 632 nodes merge different values, e.g. a monument number and a city population of 10,057. With text and kind, there are no such merges.

**Geometries**:

In [4]:
print(r.geometry_counts.to_dict())
print("Geometry types:", r.geometries["geometry_type"].value_counts().to_dict())
print(f"Entities with their own geometry: {len(r.locations):,}")

{'WKT literals': 21208, 'Not valid WKT': 0, 'Empty': 0, 'Usable': 21208}
Geometry types: {'Polygon': 11215, 'Point': 9893, 'MultiPolygon': 100}
Entities with their own geometry: 31,169


All 21,208 WKT literals are valid. 31,169 entities have their own geometry; all other nodes get a location only through flooding.

### 1.2 Checks

Each row is a problem found in the EDA, in the authors' pipeline and in ours. The distance row uses two points with a known distance (34.6 km).

In [5]:
r.checks

,authors' pipeline,ours,expected
check,,,
WKT literals recognised as geometries,12851.0,21208.0,"all usable of 21,208"
WKT literals with keywords not in capitals,8357.0,0.0,0
Geometry centres outside the country,NaN,0.0,0
"Amsterdam–Utrecht distance, km (true 34.6)",39.4,34.6,34.6
Self-loops,8396.0,0.0,0
Nodes merging an entity and a literal,8396.0,0.0,0
Nodes merging literals of different types,632.0,0.0,0


The authors' pipeline misses 8,357 geometries (`Point(...)`), swaps coordinates (39.4 km instead of 34.6 km) and has 8,396 self-loops. Ours has none of these problems.

## 2. Relations

One row per relation: edges, distinct objects, entity or literal objects, and spatial spread. The spread is the median distance of located entities that share an object to their mean position, as in the EDA. A small spread means the relation links nearby things. The table only describes the relations; choosing them is part of the experiments.

In [6]:
loc = r.locations
baseline_km = pd.Series(geodesic_km(loc["lon"], loc["lat"], loc["lon"].mean(), loc["lat"].mean())).median()
print(f"Baseline: median distance of a located entity to the mean position of all of them: {baseline_km:.1f} km")
relations = r.relations.assign(relation=r.relations["relation"].map(short), literal_kind=r.relations["literal_kind"].map(lambda k: k if pd.isna(k) else short(k)))
print(f"{len(relations)} relations, {relations['median_km'].notna().sum()} with a spatial spread")
relations.dropna(subset=["median_km"]).sort_values("median_km")

Baseline: median distance of a located entity to the mean position of all of them: 75.2 km
60 relations, 29 with a spatial spread


,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
12,asWKT,20866,20837,literal,wktLiteral,54.0,0.00
11,hasGeometry,21039,20871,entity,NaN,302.0,0.00
19,street-address,8396,8368,literal,plain,46.0,0.00
18,postal-code,8396,5846,literal,plain,3904.0,0.03
45,internComplexNummers,630,490,literal,plain,224.0,0.03
44,complexnummer,630,490,literal,plain,224.0,0.03
1,description,57042,33659,literal,@nl-nl,351.0,0.07
51,neighbourhood,258,93,literal,plain,205.0,0.27
21,city,8396,1536,entity,NaN,7793.0,0.37
53,location,156,99,literal,plain,76.0,0.67


Baseline: 75 km.

- Spatial: street address, postal code, neighbourhood, `city` (0.4 km), `sfWithin` (1.9 km).
- Regional: `parentADM2` (3 km), `parentADM1` (22 km).
- Not spatial: `type`, house numbers, building years (66–88 km).

The full table, saved as `relations.csv`:

In [7]:
relations

,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
0,type,130126,7,entity,NaN,23515.0,75.90
1,description,57042,33659,literal,@nl-nl,351.0,0.07
2,depicts,47872,8396,entity,NaN,NaN,NaN
3,depiction,47872,47872,entity,NaN,NaN,NaN
4,isPartOf,47384,38,literal,@nl-nl,NaN,NaN
5,locator,46123,46091,literal,anyURI,NaN,NaN
6,thumbnail,46108,46061,literal,base64Image,NaN,NaN
7,creator,45009,626,literal,plain,NaN,NaN
8,created,44215,1081,literal,plain,NaN,NaN
9,fotograaf,43863,463,literal,plain,NaN,NaN


### Hubs

The nodes with the most edges. During flooding, hubs mix geometries from everywhere; in the EDA, 18.5% of nodes ended at one national centroid.

In [8]:
hubs = r.hubs.assign(node=r.hubs["node"].str[-60:], kind=r.hubs["kind"].map(short), main_relation=r.hubs["main_relation"].map(short))
hubs.head(15)

,node,kind,degree,in_degree,out_degree,main_relation,main_relation_share
0,http://www.opengis.net/ont/geosparql#Geometry,entity,65576,65576,0,type,1.000
1,https://data.labs.pdok.nl/rce/def/Afbeelding,entity,60996,60996,0,type,1.000
2,"""zwart wit negatief""@nl-nl",@nl-nl,23232,23232,0,techniek,1.000
3,"""Fotocollectie""@nl-nl",@nl-nl,18146,18146,0,isPartOf,1.000
4,"""Collectie gebouwd""@nl-nl",@nl-nl,16868,16868,0,isPartOf,1.000
5,"""Dukker, G.J.""",plain,13544,13544,0,creator,0.500
6,"""Wal, A.J. van der""",plain,9396,9396,0,creator,0.500
7,"""true""^^<http://www.w3.org/2001/XMLSchema#boolean>",boolean,8230,8230,0,isFree,1.000
8,"""Agrarisch Erfgoed - Fotocollectie SHBO""@nl-nl",@nl-nl,8090,8090,0,isPartOf,1.000
9,"""Digitale opname""@nl-nl",@nl-nl,6023,6023,0,techniek,1.000


The largest hubs are classes (`Geometry`, `Afbeelding`) and shared photo literals (technique, collection, photographer). None of them is a place. Saved as `hubs.csv`.

## 3. Labels and splits

The official KGBench splits. All checks should be 0.

In [9]:
display(r.splits)
r.label_checks

split,train,valid,test,total
class,,,,
0,205,38,76,319
1,2578,478,956,4012
2,387,72,144,603
3,1496,278,555,2329
4,728,135,270,1133
total,5394,1001,2001,8396


,count
Nodes labelled more than once,0
Labelled nodes missing from the graph,0
"Triples using the target relation (monumentCode, http://purl.org/dc/terms/subject)",0


8,396 monuments in 5 classes, with the same class shares in each split. No label leaks into the graph.

## 4. Saving

Outputs in `data/kgbench_dmg777k/processed/`:

- `triples.parquet`: subject, predicate, object (node keys), object_kind
- `geometries.parquet`: node, WKT, geometry type, centre (lon, lat)
- `labels.csv`: node, class, split
- `relations.csv`, `hubs.csv`: the tables from Section 2

In [10]:
pipeline.save(r)
for path in sorted(r.dataset.processed_dir.iterdir()):
    print(f"{path.name:20} {path.stat().st_size / 1e6:8.1f} MB")

geometries.parquet      105.7 MB
hubs.csv                  0.0 MB
labels.csv                0.5 MB
relations.csv             0.0 MB
triples.parquet         580.9 MB


The end of the noteboook. enjoy your processed data